# **Day 80: Memory, Mutability and Copying**

Today we understand how Python stores values in memory. Many confusing bugs become clear after this.

## **Variables are Name Tags**
A variable does not "contain" a value. It is a **name tag** stuck on an object in memory. Two name tags can be stuck on the **same** object.

In [1]:
a = [1, 2, 3]
b = a                    # b is another name tag on the SAME list
print(id(a), id(b), a is b)

b.append(4)
print(a)                 # a also shows 4, because it is the same list

1829109447744 1829109447744 True
[1, 2, 3, 4]


## **is vs ==**
- `==` checks if the **values** are equal.
- `is` checks if both names point to the **same object** in memory.

In [2]:
x = [1, 2]
y = [1, 2]
print(x == y)       # Same values
print(x is y)       # But two different lists

True
False


We should use `is` mainly to check for `None`: `if value is None:`

## **Mutable and Immutable**
| Immutable (cannot change) | Mutable (can change) |
|---|---|
| int, float, bool, str, tuple, frozenset | list, dict, set, most objects |

In [3]:
s = "kolkata"
print(id(s))
s += " city"          # Makes a NEW string
print(id(s))          # Different address

lst = [1]
print(id(lst))
lst += [2]            # Changes the SAME list
print(id(lst))        # Same address

1829109233200
1829109538928
1829109538752
1829109538752


## **How Functions Receive Values**
A function receives the **same object**, not a copy.
- If the function **changes** a mutable object (like appending to a list), the change is visible outside.
- If the function **gives the name a new value**, the outside is not affected.

In [4]:
def add_item(bag):
    bag.append("umbrella")      # Changes the object: visible outside

def replace_bag(bag):
    bag = ["new bag"]           # Only the local name changes: not visible outside

my_bag = ["phone"]
add_item(my_bag)
replace_bag(my_bag)
print(my_bag)

['phone', 'umbrella']


## **Shallow Copy vs Deep Copy**
- **Shallow copy:** Makes a new outer container, but the **inner** objects are still **shared**.
- **Deep copy:** Copies **everything**, all the way inside.

**Real-life example:** A shallow copy is like making a **photocopy of a class list** that says "Rahul's notebook is on shelf 3". Both lists point to the same notebook. A deep copy is like **copying the notebook itself**.

In [5]:
import copy

trip = {"name": "Darjeeling", "members": ["Rahul", "Priya"], "budget": {"hotel": 6000}}

shallow = copy.copy(trip)
deep = copy.deepcopy(trip)

trip["members"].append("Sagnik")        # Change an inner list
trip["budget"]["hotel"] = 9000          # Change an inner dictionary
trip["name"] = "Sikkim"                 # Change an outer value

print("Original:", trip)
print("Shallow :", shallow)
print("Deep    :", deep)

Original: {'name': 'Sikkim', 'members': ['Rahul', 'Priya', 'Sagnik'], 'budget': {'hotel': 9000}}
Shallow : {'name': 'Darjeeling', 'members': ['Rahul', 'Priya', 'Sagnik'], 'budget': {'hotel': 9000}}
Deep    : {'name': 'Darjeeling', 'members': ['Rahul', 'Priya'], 'budget': {'hotel': 6000}}


The shallow copy **shares** the inner list and dictionary, so it also shows "Sagnik" and 9000. The deep copy is fully separate.

## **Example from Remote Sensing: The Image Grid Trap**
A very common mistake when making an empty image grid (for example, a blank classified map):

In [6]:
rows, cols = 3, 4

wrong = [[0] * cols] * rows            # 3 name tags on the SAME row!
wrong[0][0] = 9
print("Wrong way:")
for r in wrong:
    print(r)

right = [[0] * cols for _ in range(rows)]    # 3 separate rows
right[0][0] = 9
print("Right way:")
for r in right:
    print(r)

Wrong way:
[9, 0, 0, 0]
[9, 0, 0, 0]
[9, 0, 0, 0]
Right way:
[9, 0, 0, 0]
[0, 0, 0, 0]
[0, 0, 0, 0]


In the wrong way, changing one pixel changed the **whole column**, because all the rows were the same list. This bug can silently ruin a classified map.

## **Reference Counting and Garbage Collection**
Python keeps count of how many name tags point to each object. When the count becomes **0**, the object is deleted and the memory is freed. This is automatic. The `gc` module (garbage collector) cleans up more complex cases.

In [7]:
import sys, gc

x = []
print(sys.getrefcount(x))      # The count includes a temporary tag used by getrefcount itself
y = x
print(sys.getrefcount(x))
del y
print(sys.getrefcount(x))
print("Objects cleaned by gc:", gc.collect())

2
3
2
Objects cleaned by gc: 3


## **__slots__: Saving Memory for Many Objects**
If we make **millions** of small objects (for example, one for every GPS point), `__slots__` can save a lot of memory.

In [8]:
import tracemalloc

class GPSPoint:
    def __init__(self, lat, lon):
        self.lat, self.lon = lat, lon

class GPSPointSlots:
    __slots__ = ("lat", "lon")
    def __init__(self, lat, lon):
        self.lat, self.lon = lat, lon

tracemalloc.start()
normal = [GPSPoint(22.5, 88.3) for _ in range(100_000)]
normal_mb = tracemalloc.get_traced_memory()[0] / 1e6
tracemalloc.stop()

tracemalloc.start()
slotted = [GPSPointSlots(22.5, 88.3) for _ in range(100_000)]
slotted_mb = tracemalloc.get_traced_memory()[0] / 1e6
tracemalloc.stop()

print(f"Normal class : {normal_mb:.1f} MB")
print(f"With __slots__: {slotted_mb:.1f} MB")

Normal class : 9.6 MB
With __slots__: 5.6 MB


## **Summary of Phase 4 (Day 61 to Day 80)**
| Topic | Days |
|---|---|
| Type hints, regex, closures, unpacking | 61 to 65 |
| Environments, packages, testing | 66 to 69 |
| Threads, processes, asyncio | 70 to 73 |
| Time zones, SQLite, web APIs, scraping | 74 to 77 |
| Command line tools, speed, memory | 78 to 80 |

In Phase 5, we learn the most important libraries for data and GIS: **NumPy, Pandas and Matplotlib**, then work with real **Remote Sensing and GIS** projects.

In [9]:
# Quick question
# After b = a.copy() for a list of lists, will changing b[0][0] change a?